In [1]:
try:
    x = 10 / 0
except ZeroDivisionError as e:
    x = 0
    print("Cannot divide by 0, result defaults to 0.")
    
print(x)

Cannot divide by 0, result defaults to 0.
0


In [9]:
try:
    result = 10 / int(input())
    print(result)
except ValueError:
    print("Not a number!")
except ZeroDivisionError:
    print("Can't divide by zero!")

Can't divide by zero!


In [7]:
try:
    result = 10 / int(input())
except (ValueError, ZeroDivisionError):
    print("Something went wrong with the math!")

Something went wrong with the math!


In [ ]:
try:
    age = int(input())
    print(age)
except ValueError as error:
    print(f"Conversion failed: {error}")

Conversion failed: invalid literal for int() with base 10: 'a'


In [11]:
try:
    numbers = [1, 2, 3]
    print(numbers[5])
except IndexError as e:
    print(f"Oops: {e}")

Oops: list index out of range


In [12]:
try:
    try:
        x = int("abc")
    except ValueError:
        print("Inner catch")
        raise
except ValueError:
    print("Outer catch")
print("Done")

Inner catch
Outer catch
Done


In [ ]:
# Exception Handling practice for User registration systems

"""Build a user registration system with these requirements:

1. Function: register_user(username, email, password, age)
2. Validation rules:
    - Username: must be 3-20 characters, only letters/numbers
    - Email: must contain "@"
    - Password: must be 8+ characters
    - Age: must be 18 or older

3. External operations: Save to database (might fail)
4. Create custom exceptions for each validation error
5. Use professional patterns: logging + raise from None
6. Handle at two levels: internal validation + API endpoint

Your Task:
Write the complete code including:
- Custom exception classes
- Internal register_user() function
- Web API endpoint that calls it
- Proper error handling, logging, and status codes"""

import logging
import psycopg2
import json
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

DB_NAME = "somedbname"
DB_USER_NAME = "somedbuser"
DB_PASSWORD = "somedbpassword"
DB_HOST = "localhost"
DB_USERS_TABLE = "users"

        
class UserValidationError(Exception):
    def __init__(self, message):
        self.message = message
        super().__init__(self.message)

class ServiceError(Exception):
    def __init__(self, message):
        self.message = message
        super().__init__(f"Service Error: {self.message}")



def register_user(username: str, email: str, password: str, age: int):

    if len(username) not in range(3, 21):
        logging.warning("Username validation failed: not in the required length (%d characters)", len(username))
        raise UserValidationError("Username must be at lease 3 characters and no more than 20 characters")
    
    if not username.isalnum():
        logging.warning("Username validation failed: contains invalid characters")
        raise UserValidationError("Username must only contain letters or numbers")
    
    if not '@' in email:
        logging.warning("Email validation failed: '%s'", email)
        raise UserValidationError("Email must contain '@'")
    
    if len(password) < 8:
        logging.warning("Password validation failed: too short")
        raise UserValidationError("Password must be at least 8 characters")

    if age < 18:
        logging.warning("Age validation failed: less than or equal to 18 (%d)", age)
        raise UserValidationError("Age must be 18 or older")
    
    try:
        with psycopg2.connect(dbname=DB_NAME, user=DB_USER_NAME, host=DB_HOST) as conn:
            with conn.cursor() as cursor:
                cursor.execute("INSERT INTO users (username, email, password, age) VALUES (%s, %s, %s, %s)", (username, email, password, age))
                conn.commit()
            
    except psycopg2.Error as e:
        logging.error("Database error during registration: %s", str(e))
        raise ServiceError("Registration temporarily unavailable") from None
    except Exception as e:
        logging.error("Unexpected Error: %s", str(e))
        raise ServiceError("Internal error") from None

app = FastAPI()
   
class UserRegister(BaseModel):  # Define data shape for automatic input validation
    username: str
    email: str
    password: str
    age: int 

@app.post('/register')
def register_endpoint(user: UserRegister):  # FastAPI auto-parses JSON → UserRegister object
    try:
        register_user(user.username, user.email, user.password, user.age)
        logging.info("New user registered successfully")
    except ServiceError as e:
        logging.error("error: %s", str(e))
        raise HTTPException(status_code=500, detail=str(e))
    except UserValidationError as e:
        logging.warning("error: %s", str(e))
        raise HTTPException(status_code=400, detail=str(e))

IndentationError: expected an indented block after 'except' statement on line 100 (2289127664.py, line 101)

In [ ]:
import json
import logging

def load_user_settings(user_id):
    """Load user settings from file, create defaults if missing"""
    filename = f"user_{user_id}.json"
    
    # TODO: Try to load existing settings
    # If file doesn't exist, return default settings:
    # {"theme": "dark", "notifications": True, "language": "en"}
    # If JSON is corrupted, print error and return defaults
    defaults = {"theme": "dark", "notifications": True, "language": "en"}
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            settings = json.load(f)
            return settings
    except FileNotFoundError as e:
        logging.error("File not found: %s", str(e))
        return defaults
    except json.JSONDecodeError as e:
        logging.error("Json Serialization failed: %s", str(e))
        return defaults
        
    
def save_user_settings(user_id, settings):
    """Save user settings to file with pretty formatting"""
    filename = f"user_{user_id}.json"
    
    # TODO: Save settings to file with 2-space indentation
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(settings, f, indent=2)
    except TypeError as e:
        logging.error("Settings contain non-serializable objects: %s", str(e))
        raise
    except OSError as e:
        logging.error("Could not write file: %s", str(e))
        raise
        
            

# Test your functions
settings = load_user_settings(123)
settings["theme"] = "light"
save_user_settings(123, settings)

ERROR:root:File not found: [Errno 2] No such file or directory: 'user_123.json'


In [ ]:
# Python function is a first-class object

# It can be assigned to a variable
def greeting():
    return "Hello World"

my_function = greeting
my_function()

# It can be passed as an argument and returned from other functions
def call_function(func):
    return func()

call_function(greeting)

'Hello World'

In [6]:
def add(a, b):
    return a + b

calculator = add
print(calculator(3, 5))

8


In [7]:
def outer_function(message):
    def inner_function():
        print(f"Inner says: {message}")
        
    return inner_function

my_func = outer_function("Hello from outer!")
my_func()

Inner says: Hello from outer!


In [8]:
def outer_function(message):
    def inner_function():
        return f"Inner says: {message}"
    
    return inner_function

my_func = outer_function("Hello from outer!")
my_func()

'Inner says: Hello from outer!'

In [ ]:
def make_multiplier(x):
    def multiply(y):
        return x * y
    
    return multiply

double = make_multiplier(2) 
# assigning `double` to `make_multiplier(2)`, double will take what the make_multiplier(2) returns, which is the `multiply` function (double = multiply)
# Since `multiply` has one parameter, we have to give an argument to the `double` function as well.
double(5)

10

In [ ]:
def make_multiplier(x):
    def multiply(y):
        return x * y
    
    def addition(y):
        return x + y
    
    return multiply, addition # return a tuple of two functions

multiply_func, add_func = make_multiplier(2) # we have to assign two variables since `make_multiplier` returns a tuple of two functions
print(multiply_func(5))
print(add_func(5))

10
7


In [13]:
# Decorator function "wraps" the original function with extra behaviors

def my_decorator(func):
    def wrapper():
        print("Before the function")
        func()
        print("After the function")
    return wrapper

def say_hello():
    print("Hello!")

decorated_hello = my_decorator(say_hello)
decorated_hello()

Before the function
Hello!
After the function


In [14]:
def shout_decorator(func):
    def wrapper():
        print("ATTENTION!")
        func()
        print("THAT'S ALL!")
        
    return wrapper

@shout_decorator
def whisper():
    print("psst... secret message")
    
whisper()

ATTENTION!
psst... secret message
THAT'S ALL!


In [ ]:
# Decorator functions with arguments

def my_decorator(func):
    def wrapper(*arg, **kwargs):
        print("Before function")
        result = func(*arg, **kwargs)
        print("After function")
        return result
    return wrapper

@my_decorator
def add(a, b):
    return a + b

print(add(3, 5))
"""OUTPUT
Before function
After function
8

Not in order!!!
"""

Before function
After function
8


In [16]:
def broken_decorator(func):
    def wrapper(*args, **kwargs):
        print("Starting calculation")
        func(*args, **kwargs)  # Missing something here
        print("Calculation done")
    return wrapper

@broken_decorator
def multiply(x, y):
    return x * y

result = multiply(4, 3)
print(f"Result: {result}")

Starting calculation
Calculation done
Result: None


In [19]:
# Use functools.wraps decorator inside a decorator to preserve the original function's name and docstring

import functools

def my_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def add(a, b):
    """Adds two numbers together"""
    return a + b

print(add.__name__)
print(add.__doc__)

add
Adds two numbers together


In [22]:
# Decorator functions with parameters: Decorator Factories

import functools

def repeat(times):
    def my_decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return my_decorator

@repeat(3)
def greet(name):
    print(f"Hello {name}!")
    
greet("Alice")

Hello Alice!
Hello Alice!
Hello Alice!


In [29]:
# Multiple or Stacking Decorators (effects are from closet to the furthest)

import functools

def bold(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return f"**{result}**"
    return wrapper

def italic(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return f"*{result}*"
    return wrapper

@bold
@italic
def greet(name):
    return f"Hello {name}!"

print(greet("Alice"))

***Hello Alice!***


In [ ]:
"""Your Task:
Create a decorator called @monitor that:

1. Takes an argument log_level (either "basic" or "detailed")
2. Times how long the function takes to execute
3. Logs different information based on the log_level:
    - "basic": Just prints function name and execution time
    - "detailed": Prints function name, arguments, return value, and execution time
4. Works with any function (handles arguments properly)
5. Preserves the original function's metadata

Requirements:
- Use time.time() for timing
- Use proper decorator factory structure
- Use functools.wraps
- Handle *args and **kwargs"""

import functools
from time import time

def monitor(log_level):
    if log_level not in ["basic", "detailed"]:
        raise ValueError(f"Invalid log_level: {log_level}. Must be 'basic' or detailed'")
    
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            start_time = time()
            result = func(*args, **kwargs)
            exec_time = time() - start_time
            
            if log_level == 'basic':
                print(f"{func.__name__} executed in {exec_time:.6f} seconds")
            elif log_level == 'detailed':
                print(f"{func.__name__} called with args: {args}, kwargs: {kwargs}, returned: {result}, executed in {exec_time:.6f} seconds")
                return result

            return result
        return wrapper
    return decorator

@monitor("basic")
def add(a, b):
    """Adds two numbers."""
    return a + b

@monitor("detailed")
def multiply(x, y):
    """Multiplies two numbers."""
    return x * y

add(3, 5)
multiply(4, 6)

add executed in 0.000001 seconds
multiply called with args: (4, 6), kwargs: {}, returned: 24, executed in 0.000001 seconds


24

In [39]:
# Async functions

import asyncio

# NON-BLOCKING - other work can happen during waits
async def non_blocking_example(num: int):
    print(f"Starting task {num}")
    await asyncio.sleep(2)  # Program can do other things during this wait
    print(f"Task {num} done")
    
# We can run multiple non-blocking tasks simultaneously
async def main():
    await asyncio.gather(
        non_blocking_example(1),
        non_blocking_example(2)
    )
    
# asyncio.run(main())  # this runs in a normal python file, but will crash in jupyter notebooks or IPython, which already has an event loop running in the background. `asyncio.run()` tries to create a new event loop, but we can't have two event loops at the same time.
await main()

Starting task 1
Starting task 2
Task 1 done
Task 2 done


In [40]:
import asyncio

async def fetch_data(name, delay):
    print(f"{name}: Starting to fetch data")
    await asyncio.sleep(delay)  # Pause here - other coroutines can run
    print(f"{name}: Data fetched!")
    return f"Data from {name}"

async def main():
    # Start both coroutines
    result1 = await fetch_data("Task A", 2)
    result2 = await fetch_data("Task B", 1)
    print(f"Results: {result1}, {result2}")

await main()

Task A: Starting to fetch data
Task A: Data fetched!
Task B: Starting to fetch data
Task B: Data fetched!
Results: Data from Task A, Data from Task B


In [41]:
async def good_task():
    for i in range(5):
        print(f"Working on item {i}")
        await asyncio.sleep(0.1)  # Yields control every iteration
        # Other tasks can run during this sleep

async def main():
    await asyncio.gather(
        good_task(),
        good_task()  
    )
    # Both tasks will interleave nicely

await main()

Working on item 0
Working on item 0
Working on item 1
Working on item 1
Working on item 2
Working on item 2
Working on item 3
Working on item 3
Working on item 4
Working on item 4


In [ ]:
import asyncio
import random

async def fetch_data(name):
    delay = random.uniform(1, 4)  # Random delay between 1-4 seconds
    print(f"{name}: Staill take rting (w{delay:.1f}s)")
    await asyncio.sleep(delay)
    print(f"{name}: Completed!")
    return f"Data from {name}"

async def main():
    tasks = [
        fetch_data("Task A"),
        fetch_data("Task B"), 
        fetch_data("Task C"),
        fetch_data("Task D")
    ]
    
    # Process results as they complete (not in original order)
    for completed_task in asyncio.as_completed(tasks):
        result = await completed_task
        print(f"Got result: {result}")
        # You can process this result immediately!

await main()

Task A: Starting (will take 3.1s)
Task B: Starting (will take 1.4s)
Task D: Starting (will take 2.8s)
Task C: Starting (will take 2.4s)
Task B: Completed!
Got result: Data from Task B
Task C: Completed!
Got result: Data from Task C
Task D: Completed!
Got result: Data from Task D
Task A: Completed!
Got result: Data from Task A


In [44]:
# Error handling with Async Functions with return_exceptions=True

async def task_that_may_fails(name):
    await asyncio.sleep(1)
    if name == "Bad Task":
        raise ValueError(f"{name} failed!")
    return f"{name} succeeded"

async def main_1():
    # without return_exceptions=True, gather() stops on first exception (default behavior)
    try:
        results = await asyncio.gather(
            task_that_may_fails("Good Task"),
            task_that_may_fails("Bad Task"),
            task_that_may_fails("Another Good Task"),
        )
    except ValueError as e:
        print(f"One task failed: {e}")
        # The other tasks are cancelled!
        
async def main_2():
    results = await asyncio.gather(
        task_that_may_fails("Good Task"),
        task_that_may_fails("Bad Task"),
        task_that_may_fails("Another Good Task"),
        return_exceptions=True
    )
    
    for i, result in enumerate(results):
        if isinstance(result, Exception):
            print(f"Task {i} failed: {result}")
        else:
            print(f"Task {i} succeeded: {result}")
            
# await main_1()
await main_2()

Task 0 succeeded: Good Task succeeded
Task 1 failed: Bad Task failed!
Task 2 succeeded: Another Good Task succeeded


In [47]:
# Timeout in async functions

async def slow_operation():
    print("Starting slow operation...")
    await asyncio.sleep(10)  # Takes 10 seconds
    return "Operation completed"

async def main():
    try:
        result = await asyncio.wait_for(slow_operation(), timeout=3.0)
        print(f"Result: {result}")
    except asyncio.TimeoutError:
        print(f"Operation timed out!")
        
await main()

Starting slow operation...
Operation timed out!


In [48]:
# Task cancellation

async def long_task():
    try:
        for i in range(10):
            print(f"Working ... step {i}")
            await asyncio.sleep(1)
        return "Task completed"
    except asyncio.CancelledError:
        print("Task was cancelled!")
        # More clean up work if any. This is the reason why this exception is used-to have a clean up process
        raise  # Re-raise to properly handle cancellation
    
async def main():
    # Start the task
    task = asyncio.create_task(long_task())
    
    # Let it run for 3 seconds
    await asyncio.sleep(3)
    
    # Cancel it
    task.cancel()
    
    try:
        result = await task
    except asyncio.CancelledError:
        print("Confirmed task was cancelled")
        
await main()

Working ... step 0
Working ... step 1
Working ... step 2
Task was cancelled!
Confirmed task was cancelled


In [52]:
# asyncio.wait() for complex task management

# Timeout example

async def worker(name, duration):
    print(f"{name}: Starting ({duration}s)")
    await asyncio.sleep(duration)
    print(f"{name}: Completed")
    return f"Result from {name}"

async def main():
    tasks = [
        asyncio.create_task(worker("Fast", 1)),
        asyncio.create_task(worker("Medium", 3)),
        asyncio.create_task(worker("Slow", 6)),
    ]
    
    # Wait maximum 4 seconds
    done, pending = await asyncio.wait(tasks, timeout=4.0)
    
    print(f"Completed tasks: {len(done)}")
    print(f"Still running: {len(pending)}")
    
    # Get results from completed tasks
    for task in done:
        result = await task
        print(f"Got: {result}")
        
    # Cancel remaining tasks
    for task in pending:
        print(f"Cancelling {task}")
        task.cancel()
        
await main()

Fast: Starting (1s)
Medium: Starting (3s)
Slow: Starting (6s)
Fast: Completed
Medium: Completed
Completed tasks: 2
Still running: 1
Got: Result from Fast
Got: Result from Medium
Cancelling <Task pending name='Task-45' coro=<worker() running at /var/folders/1g/qthldwlx4jjdlv4ghhpylybm0000gn/T/ipykernel_5559/2147593024.py:7> wait_for=<Future pending cb=[Task.task_wakeup()]>>
